# Test de génération de synonymes cliniques « vie réelle » via Mistral Large

## Contexte

Phase 1 du chantier d'enrichissement du CSV avec des formulations cliniques
générées par LLM à partir des fiches existantes. Objectif final : avoir des
formulations correspondant à ce que les cliniciens écrivent dans les
comptes-rendus d'hospitalisation (CRH) français, en complément des sources
actuelles (Index CIM-10 vol3, AP-HP, ORPHANET, etc.).

Le dictionnaire **CepiDc 2015** (147 340 entrées pour ~6 300 codes) sert de
référence « vie réelle » pour calibrer le LLM via du few-shot prompting.

## Phase 1 (ce notebook)

- Test du prompt sur 3 codes témoins (A18.1, J18.8, R51)
- API Mistral standard (pas batch, pour itération rapide)
- ~10 secondes d'exécution totale
- Coût négligeable (~0,03 €)

## Phase 2 (chantier ultérieur)

- Génération massive sur ~16 000 codes via API batch Mistral. Hors scope ici.

## Pré-requis

1. `uv sync --extra llm` pour installer `mistralai`
2. Copier `config/secrets.yaml.example` → `config/secrets.yaml` et y mettre la clé Mistral

In [12]:
from __future__ import annotations

import json
import random
import re
import time
from pathlib import Path

import polars as pl
import yaml
from mistralai.client import Mistral

In [13]:
# Détection robuste de la racine projet (pour rendre les chemins
# absolus indépendamment du cwd du kernel Jupyter).
def _find_project_root(start: Path | None = None) -> Path:
    """Remonte depuis `start` (ou cwd) jusqu'à trouver `pyproject.toml`."""
    p = (start or Path.cwd()).resolve()
    while p != p.parent:
        if (p / "pyproject.toml").is_file():
            return p
        p = p.parent
    raise RuntimeError("pyproject.toml introuvable depuis cwd")


PROJECT_ROOT = _find_project_root()

# Chemins (absolus)
CEPIDC_PATH = PROJECT_ROOT / "data/CIM_CEPIDC_2015/CepiDc_Dictionnaire2015.csv"
CARDS_DIR = PROJECT_ROOT / "outputs/cards_library"
SECRETS_PATH = PROJECT_ROOT / "config/secrets.yaml"
OUTPUT_PATH = PROJECT_ROOT / "outputs/llm_synonymes_test.jsonl"

# LLM
MODEL = "mistral-large-latest"
TEMPERATURE = 0.5
MAX_TOKENS = 1500
N_TARGET = 20
MAX_CEPIDC_EXAMPLES = 5

# Test
CODES_TEMOINS = ["A18.1", "J18.8", "R51"]
SEED = 42

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"CARDS_DIR    : {CARDS_DIR.is_dir()} → {CARDS_DIR}")
print(f"CEPIDC_PATH  : {CEPIDC_PATH.is_file()} → {CEPIDC_PATH}")


PROJECT_ROOT : /Users/remi/Documents/recode-icd-merge/recode-icd
CARDS_DIR    : True → /Users/remi/Documents/recode-icd-merge/recode-icd/outputs/cards_library
CEPIDC_PATH  : True → /Users/remi/Documents/recode-icd-merge/recode-icd/data/CIM_CEPIDC_2015/CepiDc_Dictionnaire2015.csv


In [14]:
# Chargement et préparation du dictionnaire CepiDc.
# - Séparateur `;`, encodage utf-8
# - Filtrage : Icd1 non nul
# - Conversion `A181` → `A18.1` (point après le 3e caractère pour les codes >= 4 car)


def icd_to_dotted(icd: str | None) -> str | None:
    """A181 → A18.1, R51 → R51, X44 → X44."""
    if icd is None:
        return None
    if len(icd) <= 3:
        return icd
    return f"{icd[:3]}.{icd[3:]}"


cepidc = (
    pl.read_csv(CEPIDC_PATH, separator=";", infer_schema_length=10_000)
    .filter(pl.col("Icd1").is_not_null())
    .with_columns(
        pl.col("Icd1")
        .map_elements(icd_to_dotted, return_dtype=pl.String)
        .alias("code_dotted")
    )
    .select("code_dotted", "DiagnosisText")
)
print(f"CepiDc chargé : {cepidc.height:,} lignes / {cepidc['code_dotted'].n_unique():,} codes uniques")
for code in CODES_TEMOINS:
    n = cepidc.filter(pl.col("code_dotted") == code).height
    print(f"  {code:6s} : {n:3d} formulations CepiDc")

CepiDc chargé : 147,340 lignes / 6,294 codes uniques
  A18.1  :  27 formulations CepiDc
  J18.8  :  74 formulations CepiDc
  R51    :  20 formulations CepiDc


In [15]:
# Helper : chargement d'une fiche markdown depuis cards_library/.
# Les fiches sont rangées dans des sous-dossiers par chapitre romain.


def load_card(code: str, cards_dir: Path = CARDS_DIR) -> str | None:
    """Cherche `<code>.md` dans les sous-dossiers de `cards_dir`. None si absent."""
    for chap_dir in cards_dir.iterdir():
        if not chap_dir.is_dir():
            continue
        target = chap_dir / f"{code}.md"
        if target.is_file():
            return target.read_text(encoding="utf-8")
    return None


def extract_libelle_from_card(card: str) -> str:
    """Extrait le libellé après `# CODE — ` dans le titre de la fiche."""
    m = re.search(r"^# \S+ — (.+)$", card, re.MULTILINE)
    return m.group(1) if m else ""


def get_cepidc_examples(
    code: str,
    cepidc_df: pl.DataFrame,
    max_n: int = MAX_CEPIDC_EXAMPLES,
    rng: random.Random | None = None,
) -> list[str]:
    """Renvoie jusqu'à `max_n` DiagnosisText pour `code` ; sample reproductible si plus."""
    sub = cepidc_df.filter(pl.col("code_dotted") == code)
    texts = sub["DiagnosisText"].drop_nulls().to_list()
    if len(texts) > max_n:
        rng = rng or random.Random(SEED)
        texts = rng.sample(texts, max_n)
    return texts


# Spot-check : libellé + nb d'exemples CepiDc pour chaque code témoin
for code in CODES_TEMOINS:
    card = load_card(code)
    if card is None:
        print(f"⚠ fiche {code} introuvable")
        continue
    lib = extract_libelle_from_card(card)
    examples = get_cepidc_examples(code, cepidc)
    print(f"{code:6s} ({len(card):5d} chars) — {lib}")
    print(f"   {len(examples)} exemples CepiDc : {examples[:3]}{'...' if len(examples) > 3 else ''}")

A18.1  ( 2307 chars) — Tuberculose de l'appareil génito-urinaire
   5 exemples CepiDc : ['tuberculose urogénitale', 'BK urinaire', 'abcès pelvien tuberculeux']...
J18.8  ( 1910 chars) — Autres pneumopathies, microorganisme non précisé
   5 exemples CepiDc : ['pleuropneumonie purulente', 'infection bronchopulmonaire germe multirésistant', 'pleuro-pneumopathie infectieuse bilatérale']...
R51    (  882 chars) — Céphalée
   5 exemples CepiDc : ['céphalée aigüe tempe', 'algies face', 'céphalées aigües']...


In [16]:
# System prompt : rôle et mission du LLM.

SYSTEM_PROMPT = """Tu es un médecin codeur français expérimenté, spécialiste de la CIM-10
et de la pratique clinique hospitalière en France. Tu as lu des
milliers de comptes-rendus d'hospitalisation (CRH) rédigés par des
cliniciens et tu connais parfaitement le langage télégraphique, les
abréviations et les raccourcis qu'ils utilisent dans leur pratique
quotidienne.

# Contexte : le codage CIM-10

Le codage CIM-10 consiste à identifier dans un document médical (CRH,
notes d'évolution, comptes-rendus opératoires) les maladies,
symptômes, états cliniques et actes documentés par les médecins, et à
leur attribuer des codes standardisés. C'est une tâche de
standardisation d'information, pas une tâche de décision médicale :
on traduit ce que le clinicien a écrit en codes normalisés.

# Ta mission : la tâche inverse

Ta mission est de réaliser la tâche inverse : à partir d'un code
CIM-10 et de sa fiche descriptive officielle, générer les formulations
cliniques que des médecins auraient pu écrire dans un CRH pour aboutir
à ce code. Tu pars donc du concept standardisé et tu retournes vers la
diversité réelle de l'écriture médicale.

Ces formulations serviront à entraîner un outil d'aide au codage
automatique : il est essentiel qu'elles soient à la fois réalistes
(elles existent dans des CRH réels) et discriminantes (elles
permettent d'identifier le code).

# La nature variable des codes CIM-10

Les codes CIM-10 n'ont pas tous la même précision clinique. Certains
désignent une entité nette et unique (ex. I10, hypertension artérielle
essentielle). D'autres, en particulier les sous-catégories terminées
par .8 (« autres formes précisées »), regroupent plusieurs entités
hétérogènes mais identifiées sous un même libellé (ex. B17.8, « autres
hépatites virales aiguës précisées »). Le libellé officiel d'un code
ne suffit donc pas toujours à savoir quelle pathologie réelle décrire.

Cas particulier des codes « sans précision » (le plus souvent terminés
par .9) : ces codes ne sont employés que lorsque le dossier ne
contenait pas l'information détaillée qui aurait permis de choisir un
code plus spécifique. Leur présence est donc un signal positif
d'incertitude : on sait qu'il y a une pathologie de cette catégorie,
mais on ignore volontairement son détail.

En conséquence, quand tu génères des formulations pour un tel code,
reste délibérément vague. Ajouter une précision que le code ne porte
pas serait une erreur, car cela contredirait la raison même pour
laquelle ce code « sans précision » a été retenu plutôt qu'un code
plus fin. À l'inverse, un détail clinique inventé devrait toujours
justifier un code plus spécifique : son absence ici est intentionnelle.

Cas particulier des cancers : la CIM-10 a pour eux une approche
essentiellement anatomique. Le niveau de détail du code par rapport à
la catégorie est donc principalement une précision anatomique.

# Utilisation de la fiche descriptive

Pour t'aider, la fiche descriptive du code contient des informations
complémentaires issues de la classification et de thésaurus médicaux :

- **Synonymes et inclusions** : formulations cliniques alternatives
  et sous-types couverts par le code. Utilise-les pour choisir un
  vocabulaire naturel et varié, sans recopier le libellé officiel.
- **Exclusions** : entités voisines qui ne relèvent pas de ce code.
  Elles délimitent le périmètre. N'oriente jamais tes formulations
  vers une entité exclue.

Reste strictement dans le périmètre du code : reformule librement à
l'intérieur de ce que couvrent libellé + inclusions + synonymes, mais
n'en sors pas. Pour les codes « sans précision », ce périmètre est
volontairement large et flou — respecte ce flou."""

In [ ]:
# User prompt : construction paramétrée.
# Bloc CepiDc varie selon présence/absence d'exemples.


_CEPIDC_WITH_EXAMPLES = """## Formulations existantes pour ce code (à NE PAS reproduire)

Les formulations suivantes sont déjà connues pour ce code. Elles te donnent le style attendu mais tu dois générer des formulations **complémentaires**, pas les reproduire :

{LIST}

Génère des formulations qui couvrent d'autres facettes du code, en gardant le même style télégraphique et clinique."""

_CEPIDC_WITHOUT_EXAMPLES = """## Formulations existantes pour ce code

Aucune formulation de référence n'est fournie pour ce code. Appuie-toi sur la fiche descriptive ci-dessus et sur ta connaissance de la pratique clinique française pour générer les formulations."""


def build_user_prompt(
    code: str,
    libelle: str,
    fiche: str,
    cepidc_examples: list[str],
    n_target: int = N_TARGET,
) -> str:
    if cepidc_examples:
        bloc = _CEPIDC_WITH_EXAMPLES.format(
            LIST="\n".join(f"- {e}" for e in cepidc_examples)
        )
    else:
        bloc = _CEPIDC_WITHOUT_EXAMPLES
    return f"""# Mission

Génère des formulations cliniques distinctes qui pourraient être effectivement présentes dans un compte-rendu d'hospitalisation français pour le code CIM-10 ci-dessous.

# Critères de validité

Une formulation est valide si et seulement si elle satisfait les 4 critères suivants :

1. **Réalisme CRH** : tu pourrais l'avoir lue telle quelle dans un CRH français récent rédigé par un clinicien (pas par un codeur ni un archiviste).

2. **Diversité de longueur** : génère un mélange équilibré de
   formulations courtes (1-3 mots, environ la moitié) et de
   formulations plus détaillées (4-8 mots, environ la moitié).
   Aucune formulation ne dépasse 8 mots.
3. **Vocabulaire clinique uniquement** : pas de termes spécifiques au vocabulaire de codage CIM-10 comme "sans précision", "SAI", "non classé ailleurs", "nca", "siège non précisé", etc. Ces mots n'apparaissent jamais dans un CRH écrit par un clinicien.

4. **Discrimination tolérante** : la formulation doit permettre d'identifier ce code spécifique. Une ambiguïté est acceptable si elle est typique de la pratique clinique réelle (par exemple, "pneumopathie" peut renvoyer à plusieurs codes, mais c'est le mot réellement utilisé en pratique).

# Type de formulations attendues

Variété attendue, classée du plus court au plus détaillé :

- **Abréviations courantes** : IDM, BPCO, AVC, BK, SCA, OAP, PTH, SARM, etc.
- **Formulations télégraphiques** : structure médicale rapide, sans articles ni mots de liaison superflus (ex. "TB rénale", "pyélo bilatérale", "IDM antéro-septal").
- **Variations idiomatiques** : différentes manières de dire la même chose en pratique clinique (ex. "tuberculose urogénitale", "tuberculose génito-urinaire", "TB génito-urinaire").
- **Formulations longues médicalement correctes** : 5 à 8 mots,
  attendues pour environ la moitié des générations. Exemples :
  "infection pleuro-pulmonaire à germes atypiques", "pyélonéphrite
  tuberculeuse chronique bilatérale", "pneumonie communautaire à
  germe non identifié", "céphalée chronique d'allure tensionnelle".
# Code à traiter

**Code** : {code}
**Libellé officiel** : {libelle}

## Fiche descriptive complète

{fiche}

{bloc}

# Format de sortie attendu

Réponds UNIQUEMENT avec un objet JSON valide contenant deux clés :

- `"code"` : le code CIM-10 traité
- `"formulations"` : une liste de strings, chaque string étant une formulation clinique distincte

```json
{{
  "code": "{code}",
  "formulations": ["formulation 1", "formulation 2", "formulation 3"]
}}
```

Aucun commentaire avant ou après le JSON. Aucun markdown. Juste le JSON.

# Nombre cible

Génère {n_target} formulations distinctes. Si tu ne peux pas en générer {n_target} crédibles (code rare, code très spécifique, peu de matière dans la fiche), génère moins. Mieux vaut 10 formulations crédibles que 20 dont 10 sont inventées.

Si une formulation t'apparaît douteuse (tu hésites à l'avoir vraiment lue dans un CRH), ne l'inclus pas."""

In [18]:
# Setup du client Mistral à partir de config/secrets.yaml.

if not SECRETS_PATH.is_file():
    raise RuntimeError(
        f"Fichier de configuration manquant : {SECRETS_PATH}.\n"
        f"Copier config/secrets.yaml.example → config/secrets.yaml et y mettre la clé Mistral."
    )

with SECRETS_PATH.open(encoding="utf-8") as f:
    secrets = yaml.safe_load(f) or {}

api_key = (secrets.get("mistral") or {}).get("api_key")
if not api_key or api_key == "your-mistral-api-key-here":
    raise RuntimeError(
        "Clé mistral.api_key manquante ou non remplie dans config/secrets.yaml."
    )

client = Mistral(api_key=api_key)
print(f"Client Mistral prêt — modèle : {MODEL}")

Client Mistral prêt — modèle : mistral-large-latest


In [19]:
# Helper d'appel LLM avec parsing JSON robuste + retry 1 fois.


def generate_for_code(
    client: Mistral,
    code: str,
    libelle: str,
    fiche: str,
    cepidc_examples: list[str],
) -> dict:
    """Génère N_TARGET formulations pour `code` via Mistral Large.

    Retour : dict avec code, libellé, formulations, métadonnées tokens/temps,
    user_prompt utilisé + raw_response_preview (debug).
    """
    user_prompt = build_user_prompt(code, libelle, fiche, cepidc_examples, N_TARGET)
    t0 = time.perf_counter()

    resp = None
    last_exc = None
    for attempt in (1, 2):
        try:
            resp = client.chat.complete(
                model=MODEL,
                temperature=TEMPERATURE,
                max_tokens=MAX_TOKENS,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
            )
            break
        except Exception as exc:
            last_exc = exc
            if attempt == 2:
                raise
            print(f"  retry après erreur : {exc}")
            time.sleep(1)

    assert resp is not None
    raw = resp.choices[0].message.content or ""
    formulations: list[str] = []
    try:
        parsed = json.loads(raw)
        formulations = parsed.get("formulations", []) if isinstance(parsed, dict) else []
    except json.JSONDecodeError as exc:
        print(f"  ⚠ parsing JSON échoué pour {code} : {exc}")

    return {
        "code": code,
        "libelle": libelle,
        "formulations": formulations,
        "n_formulations": len(formulations),
        "n_cepidc_examples": len(cepidc_examples),
        "cepidc_examples": cepidc_examples,
        "model": MODEL,
        "temperature": TEMPERATURE,
        "prompt_tokens": resp.usage.prompt_tokens,
        "completion_tokens": resp.usage.completion_tokens,
        "elapsed_s": round(time.perf_counter() - t0, 2),
        "user_prompt": user_prompt,
        "raw_response_preview": raw[:500],
    }

In [20]:
# Test pipeline sur A18.1 seul (itération rapide).

code = "A18.1"
card = load_card(code)
assert card is not None, f"fiche {code} introuvable"
libelle = extract_libelle_from_card(card)
examples = get_cepidc_examples(code, cepidc)

print(f"Code : {code}")
print(f"Libellé : {libelle}")
print(f"Fiche : {len(card)} chars")
print(f"Exemples CepiDc : {examples}")

result_a181 = generate_for_code(client, code, libelle, card, examples)
print(f"\n→ {result_a181['n_formulations']} formulations en {result_a181['elapsed_s']}s")
print(f"  Tokens in/out : {result_a181['prompt_tokens']} / {result_a181['completion_tokens']}")
print("\nFormulations :")
for i, f in enumerate(result_a181["formulations"], 1):
    print(f"  {i:2d}. {f}")

Code : A18.1
Libellé : Tuberculose de l'appareil génito-urinaire
Fiche : 2307 chars
Exemples CepiDc : ['tuberculose urogénitale', 'BK urinaire', 'abcès pelvien tuberculeux', 'tuberculose uronéphrologique', 'tbc urinaire']

→ 20 formulations en 4.3s
  Tokens in/out : 2442 / 204

Formulations :
   1. TB rénale
   2. tuberculose vésicale
   3. BK génito-urinaire
   4. pyélonéphrite tuberculeuse
   5. TB prostatique
   6. tbc uretérale
   7. néphrite tuberculeuse
   8. orchite tuberculeuse
   9. épididymite BK
  10. cystite tuberculeuse
  11. TB génitale masculine
  12. tuberculose urétérale
  13. BK rénal
  14. salpingite tuberculeuse
  15. endométrite BK
  16. TB vésiculo-séminale
  17. hématurie tuberculeuse
  18. fistule urinaire tuberculeuse
  19. tuberculose pelvienne
  20. prostatite à BK


In [21]:
# Boucle sur les 3 codes témoins.

results: list[dict] = []
for code in CODES_TEMOINS:
    card = load_card(code)
    if card is None:
        print(f"⚠ fiche {code} introuvable — skip")
        continue
    libelle = extract_libelle_from_card(card)
    examples = get_cepidc_examples(code, cepidc)
    print(f"→ {code} ({libelle}) — {len(examples)} exemples CepiDc")
    result = generate_for_code(client, code, libelle, card, examples)
    print(
        f"   {result['n_formulations']} formulations / "
        f"{result['prompt_tokens']}+{result['completion_tokens']} tokens / "
        f"{result['elapsed_s']}s"
    )
    results.append(result)
print(f"\nTotal : {len(results)} codes traités")

→ A18.1 (Tuberculose de l'appareil génito-urinaire) — 5 exemples CepiDc
   20 formulations / 2442+205 tokens / 4.39s
→ J18.8 (Autres pneumopathies, microorganisme non précisé) — 5 exemples CepiDc
   20 formulations / 2407+250 tokens / 4.81s
→ R51 (Céphalée) — 5 exemples CepiDc
   20 formulations / 1975+191 tokens / 3.56s

Total : 3 codes traités


In [22]:
# Affichage récap + détail par code (incluant le prompt envoyé au LLM
# pour faciliter l'itération sur le wording).

print("=" * 76)
print(f"{'Code':6s} | {'n_formul':>8s} | {'tok_in':>7s} | {'tok_out':>7s} | {'elapsed':>8s}")
print("-" * 76)
tot_in = tot_out = 0
for r in results:
    print(
        f"{r['code']:6s} | {r['n_formulations']:>8d} | "
        f"{r['prompt_tokens']:>7d} | {r['completion_tokens']:>7d} | "
        f"{r['elapsed_s']:>7.2f}s"
    )
    tot_in += r["prompt_tokens"]
    tot_out += r["completion_tokens"]
print("-" * 76)
print(f"{'TOTAL':6s} | {sum(r['n_formulations'] for r in results):>8d} | {tot_in:>7d} | {tot_out:>7d}")
# Coût indicatif : Mistral Large ~2 €/M input, ~6 €/M output (à actualiser au moment de l'exécution).
cout = tot_in / 1_000_000 * 2 + tot_out / 1_000_000 * 6
print(f"Coût estimé : ~{cout:.4f} €")

# System prompt commun (affiché une fois, en haut)
print("\n" + "#" * 76)
print("# SYSTEM PROMPT (constant pour tous les codes)")
print("#" * 76)
print(SYSTEM_PROMPT)

# Détail par code : user prompt + exemples + formulations
for r in results:
    print("\n" + "=" * 76)
    print(f"{r['code']} — {r['libelle']}")
    print("=" * 76)

    print("\n--- USER PROMPT ENVOYÉ AU LLM ---")
    print(r["user_prompt"])

    print(f"\n--- Exemples CepiDc fournis ({r['n_cepidc_examples']}) ---")
    for e in r["cepidc_examples"]:
        print(f"  · {e}")

    print(f"\n--- Formulations générées ({r['n_formulations']}) ---")
    for i, f in enumerate(r["formulations"], 1):
        print(f"  {i:2d}. {f}")


Code   | n_formul |  tok_in | tok_out |  elapsed
----------------------------------------------------------------------------
A18.1  |       20 |    2442 |     205 |    4.39s
J18.8  |       20 |    2407 |     250 |    4.81s
R51    |       20 |    1975 |     191 |    3.56s
----------------------------------------------------------------------------
TOTAL  |       60 |    6824 |     646
Coût estimé : ~0.0175 €

############################################################################
# SYSTEM PROMPT (constant pour tous les codes)
############################################################################
Tu es un médecin codeur français expérimenté, spécialiste de la CIM-10
et de la pratique clinique hospitalière en France. Tu as lu des
milliers de comptes-rendus d'hospitalisation (CRH) rédigés par des
cliniciens et tu connais parfaitement le langage télégraphique, les
abréviations et les raccourcis qu'ils utilisent dans leur pratique
quotidienne.

# Contexte : le codage CIM-10

Le c

In [ ]:
# Sauvegarde JSONL (une ligne par code).

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"écrit : {OUTPUT_PATH} ({OUTPUT_PATH.stat().st_size:,} bytes, {len(results)} lignes)")